In [ ]:
!pip install -q torch transformers accelerate peft datasets

In [ ]:
!pip install -q -U torchao peft

In [ ]:
from datasets import Dataset
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# 1. Küçük örnek veri seti (Soru -> Net teknik cevap)
ornekler = [
    {
        "soru": "Sinyal kaybı 15 saniyeyi aşarsa ne olur?",
        "cevap": "Otomatik Eve Donus (RTH) proseduru baslatilir.",
    },
    {
        "soru": "Batarya voltaji 22.4V altina duserse ne yapilir?",
        "cevap": (
            "Kritik uyari verilir, faydasiz yuk enerjisi kesilir ve guc ucus"
            " kontrolune aktarilir."
        ),
    },
    {
        "soru": "Egzoz sicakligi 120 dereceyi gecerse ne olur?",
        "cevap": "Otomatik guc kisma (derating) modu devreye girer.",
    },
    {
        "soru": "Karbon-seramik fren soguma suresi nedir?",
        "cevap": "Inis sonrasi minimum 20 dakikadir.",
    },
    {
        "soru": "Buz cozucu sistem ne zaman aktiflesir?",
        "cevap": (
            "Dis sicaklik +2 derecenin altina inip nem %80'i gecince otonom"
            " aktiflesir."
        ),
    },
    {
        "soru": "Hidrolik inis takimlari hangi irtifada acilir?",
        "cevap": "Yerden 150 metre irtifada acilir.",
    },
]


def format_ve_tokenize(ornek):
  metin = (
      f"Soru: {ornek['soru']}\nCevap: {ornek['cevap']}{tokenizer.eos_token}"
  )
  tokens = tokenizer(
      metin, truncation=True, max_length=128, padding="max_length"
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens


ham_veri = Dataset.from_list(ornekler)
egitim_verisi = ham_veri.map(
    format_ve_tokenize, remove_columns=["soru", "cevap"]
)

# 2. Temel modeli yukle ve ana agirliklari dondur
base_model = AutoModelForCausalLM.from_pretrained(
    model_adi, torch_dtype=torch.float16, device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # LoRA matrislerinin rank degeri (kucuk rank = daha az parametre)
    lora_alpha=16,  # Adaptor olcekleme katsayisi
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Attention katmanlarindaki projeksiyonlara ekle
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

# 3. Kisa bir egitim dongusu
training_args = TrainingArguments(
    output_dir="./lora_iha_cikti",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=egitim_verisi,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

# 4. Egitim sonrasi test
test_prompt = "Soru: Sinyal kaybi 15 saniyeyi asarsa ne olur?\nCevap:"
girdiler = tokenizer(test_prompt, return_tensors="pt").to(lora_model.device)
with torch.no_grad():
  cikti = lora_model.generate(
      **girdiler, max_new_tokens=40, temperature=0.1, do_sample=True
  )

print(
    "\nMODEL CIKTISI:\n",
    tokenizer.decode(cikti[0], skip_special_tokens=True),
)

W1121 21:21:04.676000 8225 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 21:21:04.760000 8225 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Step,Training Loss
5,4.357109
10,4.129009
15,4.227158



MODEL CIKTISI:
 Soru: Sinyal kaybi 15 saniyeyi asarsa ne olur?
Cevap: Sıfır

Bu soru, bir elektronik sistemdeki bir sinyal kaybının nasıl gerçekleştiğini açıklar. Sınırlı süre içinde sinyal kay
